In [2]:
# rag_faiss_with_caches.py
# RAG (LangChain + FAISS) with:
#   1) Embedding cache → CacheBackedEmbeddings + LocalFileStore
#   2) LLM answer cache → langchain.llm_cache via SQLiteCache
#
# Usage:
#   pip install -U langchain langchain-openai faiss-cpu pypdf python-dotenv pandas
#   export OPENAI_API_KEY=sk-...   # (Windows: setx OPENAI_API_KEY "sk-...")
#   python rag_faiss_with_caches.py

# import os, time
# from pathlib import Path
# from dotenv import load_dotenv
# import pandas as pd

# #from langchain.chat_models import ChatOpenAI
# from langchain_openai import ChatOpenAI
# #from langchain.embeddings import OpenAIEmbeddings
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# #from langchain.vectorstores import FAISS
# #from langchain.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
# #from langchain.text_splitter import RecursiveCharacterTextSplitter

# from langchain_community.vectorstores import FAISS
# from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.runnables import RunnablePassthrough

# import langchain
# from langchain.cache import SQLiteCache
# from langchain.storage import LocalFileStore
# from langchain.embeddings import CacheBackedEmbeddings


import os, time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import langchain
from langchain_community.cache import SQLiteCache
#from langchain.storage import LocalFileStore
#from langchain_community.storage import LocalFileStore
#from langchain.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings
load_dotenv()



load_dotenv()

# ---------------------------
# Config
# ---------------------------
PERSIST_DIR     = "./faiss_db1"
EMB_CACHE_DIR   = "./emb_cache1"
LLM_CACHE_PATH  = "./.langchain_llm_cache1sqlite"
DATA_DIR        = "./data"
COLLECTION      = "docs1"
EMBED_MODEL     = "text-embedding-3-small"
CHAT_MODEL      = "gpt-4o-mini"

Path(LLM_CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)
langchain.llm_cache = SQLiteCache(database_path=LLM_CACHE_PATH)

# ---------------------------
# Helpers
# ---------------------------
def load_docs():
    """Load docs from ./data (PDF/TXT). Fall back to small samples."""
    docs = []
    data_path = Path(DATA_DIR)
    if data_path.exists():
        # Text files
        loader = DirectoryLoader(
            DATA_DIR,
            glob="**/*",
            loader_cls=TextLoader,
            show_progress=True,
            use_multithreading=True,
        )
        try:
            docs.extend(loader.load())
        except Exception:
            pass
        # PDFs
        for pdf in data_path.rglob("*.pdf"):
            try:
                docs.extend(PyPDFLoader(str(pdf)).load())
            except Exception as e1:
                print(e1)
                pass

    if not docs:
        from langchain.schema import Document
        docs = [
            Document(page_content=("LangChain is a framework for developing LLM apps. "
                                   "It integrates vector stores like FAISS and supports RAG pipelines."),
                     metadata={"source": "sample:langchain"}),
            Document(page_content=("FAISS is a library for efficient similarity search "
                                   "and clustering of dense vectors, often used for embeddings."),
                     metadata={"source": "sample:faiss"}),
        ]
    return docs


def build_or_load_faiss(cached_embeddings) -> FAISS:
    """Create/load a persistent FAISS index using cached embeddings."""
    Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    index_file = Path(PERSIST_DIR) / "faiss.index"
    store_file = Path(PERSIST_DIR) / "faiss.pkl"

    if index_file.exists() and store_file.exists():
        print("Loading existing FAISS index...")
        return FAISS.load_local(
            folder_path=PERSIST_DIR,
            embeddings=cached_embeddings,
            allow_dangerous_deserialization=True,
        )

    print("Building new FAISS index...")
    docs = load_docs()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    vs = FAISS.from_documents(chunks, cached_embeddings)
    vs.save_local(PERSIST_DIR)
    return vs


def make_rag_chain(retriever, llm):
    """Simple RAG chain: retrieve → prompt → LLM → string."""
    def format_docs(docs):
        return "\n\n".join(
            f"Source: {d.metadata.get('source','?')}\n{d.page_content}" for d in docs
        )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system",
             "You are a concise, helpful assistant. Use the provided context to answer. "
             "If the answer isn't in the context, say so.\n\nContext:\n{context}"),
            ("human", "{question}"),
        ]
    )

    chain = (
        {
            "context": retriever | (lambda docs: format_docs(docs)),
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


def timed(fn):
    def _inner(*args, **kwargs):
        t0 = time.time()
        out = fn(*args, **kwargs)
        return out, time.time() - t0
    return _inner


# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":
    # 1) Embedding cache
    Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    base_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    byte_store = LocalFileStore(EMB_CACHE_DIR)
    cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
        base_embeddings,
        byte_store,
        namespace=f"{EMBED_MODEL}-v1",
    )

    # 2) Persistent FAISS index
    vstore = build_or_load_faiss(cached_embeddings)
    retriever = vstore.as_retriever(search_kwargs={"k": 4})

    # 3) LLM (with SQLite cache)
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    # 4) RAG chain
    chain = make_rag_chain(retriever, llm)

    df = pd.read_csv(r"C:\Users\surya.adatravu\Documents\RAGAnalysis\RA_FSM_QA.csv")
    df["t0"] = 0
    df["t1"] = 0
    df["ans0"] = ''
    df['ans1'] = ''

    for i1 in range(df.shape[0]):
        question = df.loc[i1, "Question"]

        ans1, t1 = timed(chain.invoke)(question)
        df.loc[i1, 't0'] = t1
        df.loc[i1, 'ans0'] = ans1

        ans2, t2 = timed(chain.invoke)(question)
        df.loc[i1, 't1'] = t2
        df.loc[i1, 'ans1'] = ans2

    df.to_csv("results_faiss_r1.csv", index=False)

    print("\nCache locations:")
    print(f"• FAISS DB:        {Path(PERSIST_DIR).resolve()}")
    print(f"• Embedding cache: {Path(EMB_CACHE_DIR).resolve()}")
    print(f"• LLM cache (SQL): {Path(LLM_CACHE_PATH).resolve()}")


C:\Users\surya.adatravu\AppData\Local\anaconda3\envs\r1\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


Building new FAISS index...


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.72it/s]
C:\Users\surya.adatravu\AppData\Local\Temp\ipykernel_11540\817169821.py:213: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '8.479846954345703' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[i1, 't0'] = t1
C:\Users\surya.adatravu\AppData\Local\Temp\ipykernel_11540\817169821.py:217: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.06138277053833' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[i1, 't1'] = t2



Cache locations:
• FAISS DB:        C:\Users\surya.adatravu\Documents\RAGAnalysis\faiss_db1
• Embedding cache: C:\Users\surya.adatravu\Documents\RAGAnalysis\emb_cache1
• LLM cache (SQL): C:\Users\surya.adatravu\Documents\RAGAnalysis\.langchain_llm_cache1sqlite


In [4]:
!pip install -r requirements.txt

  Using cached faiss_cpu-1.12.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pypdf-6.1.3-py3-none-any.whl.metadata (7.1 kB)
  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.2 MB 4.1 MB/s eta 0:00:05
   -- ------------------------------------- 1.0/18.2 MB 2.9 MB/s eta 0:00:06
   --- ------------------------------------ 1.6/18.2 MB 2.9 MB/s eta 0:00:06
   ----- ---------------------------------- 2.4/18.2 MB 3.2 MB/s eta 0:00:05
   ------ --------------------------------- 3.1/18.2 MB 3.4 MB/s eta 0:00:05
   -------- ------------------------------- 3.9/18.2 MB 3.5 MB/s eta 0:00:05
   ---------- ----------------------------- 4.7/18.2 MB 3.6 MB/s eta 0:00:04
   ------------ --------------------------- 5.5/1